# Reproducing Moore et al. (2025) from dandiset 001699

**Paper.** Moore N, Duszkiewicz AJ, Asiminas A, Dudchenko PA, Peyrache A, Wood ER (2025). Age-dependent
alterations in the head-direction signal in a rat model of Fragile X Syndrome. *bioRxiv*.
[doi:10.1101/2025.01.09.632139](https://doi.org/10.1101/2025.01.09.632139). This attempt used the full
text of version 1 (9 January 2025), the version the dandiset links and whose title the target names. A
revised version 2 (7 January 2026, "Altered developmental profile of the cortical head-direction signal
in a rat model of Fragile X Syndrome") was also read; its Methods are used only where version 1 is
silent (the Bayesian decoder, the speed smoothing), and its numbers are compared in the attempt record.

**Data.** [DANDI:001699](https://dandiarchive.org/dandiset/001699/0.260917.2322), version
`0.260917.2322`, streamed from the archive. The paper cites no version (its Data Availability says the
data will be released at publication). Two versions are published, `0.260917.2313` and
`0.260917.2322`, nine minutes apart; the data check below shows that their assets are the same files,
and the metadata differ only in the title, description and study target. The most recent published
version is pinned, as the protocol asks when no version is cited and no earlier one matches the paper
better.

**Code.** The dandiset links the lab's package woodcode
(https://github.com/Wood-Dudchenko-lab/woodcode, read at commit
`61ee1784ef8bdeed1befa4d2dd2dd71dc8befd60`). It holds general functions (waveform trough-to-peak, Skaggs
information, Rayleigh vector length, a Poisson decoder, loading NWB files) and not the paper's analysis,
which the Methods say was run in Matlab and R. Where the Methods leave a choice open and woodcode
settles it (the channel used for trough-to-peak, positive waveforms, circular smoothing) the helper
module `hdfx.py` follows woodcode; its docstring lists where each parameter comes from.

| Claim | Paper | Tested |
|---|---|---|
| unit-counts | Results: 406, 372, 543, 507 units in 7, 5, 6, 6 rats | tested |
| units-per-recording | Results: 8 to 83 and 30 to 114 units per recording | tested |
| figs1c-juvenile-behaviour-similar | Fig. S1c | tested |
| figs2c-adult-behaviour-similar | Fig. S2c | tested |
| figs1-juvenile-cell-metrics-similar | Fig. S1f-i | tested |
| figs2-adult-cell-metrics-similar | Fig. S2f-i | tested |
| fig1ef-juvenile-sharper-tuning | Fig. 1e-f | tested |
| fig1g-juvenile-halves-correlation | Fig. 1g | tested |
| fig1h-juvenile-smaller-offset | Fig. 1h | tested |
| fig1j-juvenile-less-drift | Fig. 1i-j | tested |
| figs1j-juvenile-epochs-correlation | Fig. S1j | tested |
| figs1k-juvenile-epoch-drift-similar | Fig. S1k | tested |
| fig2c-juvenile-isomap-wake-correlation | Fig. 2c | tested |
| fig2d-juvenile-isomap-wake-sharper | Fig. 2d | tested |
| fig2g-juvenile-isomap-rem-correlation | Fig. 2e-g | tested |
| fig2h-juvenile-isomap-rem-sharper | Fig. 2h | tested |
| fig3ef-adult-tuning-similar | Fig. 3e-f | tested |
| fig3g-adult-halves-correlation | Fig. 3g | tested |
| fig3h-adult-larger-offset | Fig. 3h | tested |
| fig3j-adult-more-drift | Fig. 3i-j | tested |
| figs2j-adult-epochs-correlation | Fig. S2j | tested |
| figs2k-adult-epoch-drift-larger | Fig. S2k | tested |
| fig4c-adult-isomap-wake-correlation | Fig. 4c | tested |
| fig4d-adult-isomap-wake-similar | Fig. 4d | tested |
| fig4g-adult-isomap-rem-correlation | Fig. 4e-g | tested |
| fig4h-adult-isomap-rem-similar | Fig. 4h | tested |
| figs1d-probe-in-posub | Fig. S1d, S2d (histology) | blocked |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")       # many small fits run in parallel processes
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from scipy import stats

import hdfx

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*converge.*")
warnings.filterwarnings("ignore", message=".*Hessian.*")
warnings.filterwarnings("ignore", message=".*boundary.*")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
print("dandiset", hdfx.DANDISET_ID, "version", hdfx.DANDISET_VERSION)

In [ ]:
started = time.time()
assets = hdfx.list_assets()
results = hdfx.process_all(list(assets.asset_id), workers=4)
meta = pd.DataFrame([r["meta"] for r in results])
print(f"{len(assets)} NWB files streamed and analysed in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
with DandiAPIClient() as client:
    ds = client.get_dandiset(hdfx.DANDISET_ID, hdfx.DANDISET_VERSION)
    raw = ds.get_raw_metadata()
    summary = raw["assetsSummary"]
    print("name:", raw["name"])
    print("archive summary: subjects", summary["numberOfSubjects"], "| files", summary["numberOfFiles"],
          "| species", [s["name"] for s in summary["species"]],
          "| variables", summary["variableMeasured"])
    print("related resources:", [(r.get("name"), r.get("relation"), r.get("url")) for r in raw["relatedResource"]])
    print("about:", [(a["name"], a["schemaKey"], a["identifier"]) for a in raw.get("about", [])])
    versions, asset_age = {}, []
    for v in ("0.260917.2313", "0.260917.2322"):
        d = client.get_dandiset(hdfx.DANDISET_ID, v)
        versions[v] = set()
        for a in d.get_assets():
            md = a.get_raw_metadata()
            versions[v].add((a.path, a.size, md["digest"]["dandi:dandi-etag"]))
            if v == hdfx.DANDISET_VERSION and a.path.endswith(".nwb"):
                subj = [w for w in md.get("wasAttributedTo", []) if w.get("schemaKey") == "Participant"]
                asset_age.append({"path": a.path, "asset_metadata_age": subj[0].get("age", {}).get("value") if subj else None})
        print(f"version {v}: {len(versions[v])} assets, title {d.get_raw_metadata()['name']!r}")
    print("same assets (path, size, etag) in both published versions:",
          versions["0.260917.2313"] == versions["0.260917.2322"])
asset_age = pd.DataFrame(asset_age).sort_values("path")
print("\nage in the archive's asset metadata (derived by the archive from the date of birth):")
print(asset_age.to_string(index=False))

The two published versions hold the same 98 assets (36 NWB files and 62 videos); they differ in title
and description only, so the choice between them does not change the analysis. The archive counts 24
subjects of *Rattus norvegicus*, as the files do. Its `about` field lists "Fmr1 protein, rat" as an
anatomy term with the MeSH identifier of Fragile X Syndrome (D005600), a metadata error.

In [ ]:
cols = ["subject", "session_id", "archive_genotype", "date_of_birth", "session_start", "age_days",
        "age_field", "n_units", "n_spikes", "n_unsorted_units", "n_channels", "unit_columns"]
print(meta[cols].to_string())
print("\nsession descriptions:", meta.session_description.unique())
print("subjects:", meta.subject.nunique(), "| files:", len(meta), "| units in all files:", meta.n_units.sum())
print("files with unsorted spike times:", meta.loc[meta.n_unsorted_units > 0, ["subject", "session_id", "n_unsorted_units"]].values.tolist())
bad_dob = meta[meta.age_days < 0]
print("files whose date of birth is after the recording:", bad_dob[["subject", "session_id", "date_of_birth", "session_start"]].values.tolist())
print("\nhistology images (acquisition/histology_images) per file:")
print(meta[["subject", "session_id", "histology_images"]].to_string())
print("files with at least one histology image:", int((meta.histology_images != "").sum()), "of", len(meta))

Every file has a units table with only `spike_times`, `waveform_mean`, `electrode_group` and
`sampling_rate`: no cell type, quality metric or depth, so the paper's cell selection is recomputed from
the waveforms and spike trains. No NWB file sets the subject's `age`; the archive's asset metadata give an age
derived from the date of birth (printed above). Subjects H3023 and H3026 have dates of birth a year
after their recordings, so the archive gives them negative ages (P-366D to P-344D), a metadata error; their age group is taken from their other
properties below. The session description of every file mentions a cue rotation, which version 1 of the
paper does not describe; most files hold a third tracked session after the second, which is not used. One
unit in H3023 (session 210812) has spike times out of order, a format error; the helper sorts them. Every file holds histology images (`acquisition/histology_images`, printed above), including the
four whose asset names lack `+image`; none marks the probe tract or the postsubiculum, so
figs1d-probe-in-posub is blocked.

In [ ]:
def counts(df, age_col, geno_col):
    g = df.groupby([age_col, geno_col])
    return pd.DataFrame({"rats": g.subject.nunique(), "recordings": g.size(), "units": g.n_units.sum(),
                         "min_units": g.n_units.min(), "max_units": g.n_units.max()})

once = meta[~meta.subject.isin(hdfx.DUPLICATE_OF)]
arch = meta.copy()
arch["archive_age_group"] = arch.archive_age_group.fillna("no valid age")
print("Archive labels (age group from date of birth), every file:")
print(counts(arch, "archive_age_group", "archive_genotype").to_string())
print("\nMain grouping, each recording once (H4825 left out):")
print(counts(once, "age_group", "genotype").to_string())
print("\nSubjects whose main group differs from the archive's:")
diff = meta[(meta.genotype != meta.archive_genotype) | (meta.age_group != meta.archive_age_group)]
print(diff[["subject", "session_id", "archive_genotype", "genotype", "archive_age_group", "age_group", "age_days", "n_units", "n_channels"]].to_string())

With the archive's labels no grouping reproduces the paper's counts (406 units in 7 juvenile WT rats,
372 in 5 juvenile Fmr1-/y, 543 in 6 adult WT, 507 in 6 adult Fmr1-/y). If H3026 is a juvenile
Fmr1-/y rat (the archive says WT, and gives it a birth date after its recording) and H3029 a juvenile
WT rat (its birth date, the same as H4830's, makes it 97 days old), three of the four groups match the
paper exactly in units and rats. H3026's 83 units in one recording is also the top of the paper's range
for juveniles. H3026 and H3029 are the only rats recorded with 64-channel probes, which fits their
being a later juvenile cohort. This is consistent with the data but not proven by it; the genotype of
H3026 cannot be checked from the files. The main grouping uses it, and the claims show the archive's
labels as a variant. The archive's genotype labels are recorded as a finding.

In [ ]:
key = ["n_units", "n_spikes"]
dup = meta[meta.duplicated(key, keep=False)].sort_values(key)
print("files sharing unit and spike counts:")
print(dup[["subject", "session_id", "archive_genotype", "n_units", "n_spikes", "spike_checksum"]].to_string())
a = results[meta.index[meta.subject == "H4824"][0]]["meta"]
b = results[meta.index[meta.subject == "H4825"][0]]["meta"]
print("first 20 spike times of every unit identical:", np.array_equal(a["first_spikes"], b["first_spikes"]))
print("trough-to-peak of every unit identical:", np.allclose(a["ttp"], b["ttp"], equal_nan=True))
print("per unit, count and sum of spike times before 1248 s identical:", np.allclose(a["early_fingerprint"], b["early_fingerprint"]))
print("spikes at the file's last spike time: H4824", a["spikes_at_last_time"], "| H4825", b["spikes_at_last_time"])
print("H4824 epochs:\n", a["epochs"].round(1).to_string())
print("H4825 epochs:\n", b["epochs"].round(1).to_string())
print("H4825 tracking frames with a repeated timestamp:", b["non_increasing_frames"])
print("H4825 units with contamination above 0.5:", int(np.sum(b["contamination"] > 0.5)), "of", len(b["contamination"]),
      "| H4824:", int(np.sum(a["contamination"] > 0.5)))

H4825's file holds H4824's recording: the same 88 units with identical waveforms, spike counts and
spike times up to 1248 s (the end of H4825's first epoch), after which H4825's spike and tracking times
are shifted piece by piece onto H4825's own epoch table and clipped at its end (many spikes then share
the last spike time, which makes 87 of 88 units look contaminated). This recording is counted once, as H4824, and
H4825 has no data of its own in the dandiset. No other pair of files shares unit and spike counts.

In [ ]:
rows = []
for r in results:
    m = r["meta"]
    rows.append({"subject": m["subject"], "session": m["session_id"],
                 "epochs (s)": "; ".join(f"{e.tag}:{e.start:.0f}-{e.stop:.0f}" for e in m["epochs"].itertuples()),
                 "task epochs": m["task_epochs"],
                 "tracking blocks over 10 min (s)": "; ".join(f"{a:.0f}-{b:.0f}" for a, b in m["blocks"] if b - a >= 600),
                 "sessions used (s)": "; ".join(f"{a:.0f}-{b:.0f}" for a, b in m["sessions"]),
                 "repeated frames": m["non_increasing_frames"]})
print(pd.DataFrame(rows).to_string())

The paper describes two 20 min exploration sessions separated by 90 min of sleep. The `epochs` and
`processing/tasks` tables do not always say which interval is which: in H3001 (200202) task epoch 1 is
called wake but spans 94 min with no tracking, and tracking starts in epoch 2; in H3026 (211003) epoch
1 spans 112 min, waking and sleep together; H4823 has a single epoch for the whole recording and no
sleep task. The exploration sessions are therefore taken from the tracking itself (helper
`exploration_sessions`): the first tracking block of at least 10 min, and the first such block starting
at least 60 min later, each cut to its first 20 min, the paper's session length. This gives sessions of
18.7 to 20.0 min (H4825 aside) before and after the sleep in every recording but two, which have one
session: in H3019 210618-1 it is the session before the sleep, and in H3001 200202, where the first
epoch has no tracking, it is the session after the sleep. Juvenile files have one
to three tracking frames whose timestamp does not increase; those frames are dropped.

In [ ]:
print(meta[["subject", "session_id", "pos_unit", "px_per_cm", "pos_x_range", "pos_y_range", "hd_range",
            "hd_nan_fraction", "pos_nan_frames", "pos_y_zero_frames", "pos_x_zero_frames", "pos_y_zero_in_sessions",
            "frame_rate", "same_pos_hd_times"]].round(3).to_string())
cm = meta[meta.px_per_cm.isna()]
print(f"\nfiles in centimetres: {len(cm)}; frames with y exactly 0 per file: {cm.pos_y_zero_frames.min()} to {cm.pos_y_zero_frames.max()}; "
      f"with x exactly 0: {cm.pos_x_zero_frames.min()} to {cm.pos_x_zero_frames.max()}")

Position is stated in centimetres, and in 34 files spans about 72 to 83 cm on each axis, consistent
with the 73 cm arena seen from above. In each of these files exactly one frame has y equal to 0 and
none has x equal to 0, so the y positions were shifted to start at 0; missing tracking is stored as
NaN, not as 0.
In two files (H3015 210417 and H4823 221108) it spans 300 and 700 units, so it is in pixels although the
file says centimetres (a metadata error); their head direction runs from -pi to pi where the other files
run from 0 to 2 pi, and they hold tracking through the sleep. For these two files the scale is set so
that the x extent of the exploration tracking equals the arena's 73 cm (4.1 and 9.5 pixels per cm).
Tracking is at 40 Hz in juveniles and 40.7 Hz in adults, consistent with the Methods' 40 Hz.
H3029 is tracked at 40.7 Hz like the adults, so it was recorded on the adult rig.

In [ ]:
ttp = np.concatenate([r["meta"]["ttp"] for r in results if r["meta"]["subject"] not in hdfx.DUPLICATE_OF])
cont = np.concatenate([r["meta"]["contamination"] for r in results if r["meta"]["subject"] not in hdfx.DUPLICATE_OF])
edges = np.arange(0, 1.25, 1 / 30)
h, _ = np.histogram(ttp[~np.isnan(ttp)], edges)
print("trough-to-peak (ms), units per bin of one sample (1/30 ms):")
print(" ".join(f"{e:.2f}:{n}" for e, n in zip(edges, h)))
print(f"range {np.nanmin(ttp):.2f} to {np.nanmax(ttp):.2f} ms; positive waveforms (no duration): {int(np.isnan(ttp).sum())} of {len(ttp)}")
for age, thr in hdfx.TTP_THRESHOLD_MS.items():
    sel = np.concatenate([r["meta"]["ttp"] for r in results if r["meta"]["age_group"] == age and r["meta"]["subject"] not in hdfx.DUPLICATE_OF])
    print(f"{age}: {np.sum(sel > thr)} of {len(sel)} units above {thr} ms")
print(f"contamination score: median {np.nanmedian(cont):.3f}, range {np.nanmin(cont):.3f} to {np.nanmax(cont):.3f}; "
      f"above 0.5: {int(np.sum(cont > 0.5))} units; below 0.5: {int(np.sum(cont < 0.5))}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(ttp[~np.isnan(ttp)], edges, color="0.4")
ax[0].axvline(0.4, ls="--", c="C1", label="adult 0.4 ms")
ax[0].axvline(0.5, ls="--", c="C0", label="juvenile 0.5 ms")
ax[0].set_xlabel("trough-to-peak (ms)")
ax[0].legend()
ax[1].hist(cont, np.linspace(0, 1, 41), color="0.4")
ax[1].set_xlabel("contamination score")
plt.tight_layout()
plt.show()

The trough-to-peak durations are bimodal with the trough between 0.40 and 0.47 ms, consistent with the
Methods' thresholds (0.5 ms for juveniles, 0.4 ms for adults). The unit totals match the paper's counts
of recorded units (see unit-counts), so the files hold all curated units, not only the putative
excitatory ones. The Methods say units with a contamination score "below 0.5" were excluded;
almost every unit is below 0.5, so this is read as "above 0.5", which removes few units.

In [ ]:
# frame_counts on frames that are not contiguous (speed-filtered), on synthetic spikes
rng = np.random.default_rng(1)
t_all = np.arange(0, 600, 0.025)
keep = rng.random(len(t_all)) > 0.3            # frequent one-frame and longer gaps
t_sel = t_all[keep]
sp = np.sort(rng.uniform(0, 600, 20000))
expected = np.histogram(sp, np.r_[t_all - 0.0125, t_all[-1] + 0.0125])[0][keep]
got = hdfx.frame_counts([sp], t_sel, 0.025)[0]
print(f"spikes in the kept frames: expected {expected.sum()}, counted {int(got.sum())}; "
      f"frames with a wrong count: {int(np.sum(got != expected))} of {len(t_sel)}")
assert np.array_equal(got, expected)

`frame_counts` counts each tracking frame on its own window, so frames kept by a speed filter
need not be contiguous; on synthetic spikes it counts every spike of the kept frames. The first round
of this attempt cut frames at gaps and lost or misplaced spikes there.

In [ ]:
lag = pd.DataFrame([r["meta"].get("lag_info", [np.nan] * len(hdfx.CLOCK_LAGS)) for r in results],
                   columns=[f"{x:g}s" for x in hdfx.CLOCK_LAGS])
lag.insert(0, "session", meta.subject + " " + meta.session_id)
values = lag.iloc[:, 1:].to_numpy(float)
lag["peak_lag_s"] = [hdfx.CLOCK_LAGS[int(np.nanargmax(v))] if np.isfinite(v).any() else np.nan for v in values]
lag["zero_over_60s"] = lag["0s"] / lag[["-60s", "60s"]].max(axis=1)
print("median HD information (bits/spike) of the cells of session 1 with spikes shifted by each lag:")
print(lag.round(3).to_string())
ok = lag.dropna(subset=["0s"])
print(f"\npeak within 0.2 s of zero in {np.sum(ok.peak_lag_s.abs() <= 0.2)} of {len(ok)} recordings;"
      f" zero-lag information at least 1.5 times that at +-60 s in {np.sum(ok.zero_over_60s >= 1.5)}")
print("recordings where the clock is not established (ratio below 1.5):", ok.loc[ok.zero_over_60s < 1.5, "session"].tolist())

Spikes and tracking share a clock: with the spikes shifted against the head direction, HD information
peaks within 0.2 s of zero lag in 34 of 35 recordings and falls towards its floor at 60 s. In H3023
210812 the information is nearly flat (0.019 to 0.023 bits/spike between -2 and 2 s) and peaks at
-0.5 s. In three recordings (H3006 200314-1 and 200314-2, H3015 210416-2), whose cells are weakly
tuned, the information at zero lag is less than 1.5 times that at 60 s. For these four a shared clock
is consistent with the data but not shown.

In [ ]:
emg = pd.DataFrame([r["meta"]["emg_median"] for r in results])
emg.insert(0, "session", meta.subject + " " + meta.session_id)
sleep_s = pd.DataFrame([r["meta"]["sleep_s"] for r in results]).add_suffix("_s")
emg = pd.concat([emg, sleep_s], axis=1)
emg["rem_in_exploration"] = [r["meta"]["rem_in_exploration"] for r in results]
print(emg.round(2).to_string())
print(f"\npseudo-EMG median lower in REM than in wake in {np.sum(emg.rem < emg.wake)} of {len(emg)} files; "
      f"REM at or below NREM in {np.sum(emg.rem <= emg.nrem + 1e-9)}")
print(f"REM per file: {emg.rem_s.min():.0f} to {emg.rem_s.max():.0f} s")

The sleep stages are consistent with the pseudo-EMG: its median is lower in REM than in wake in all 36
files, and at or below its NREM median in 31. A few REM intervals overlap the tracking blocks of the
exploration sessions (counted above, up to 4 in H3015 210416-2 and 3 in H4822). The REM analyses use
every REM interval, as the claims' criterion says, and show the result with those intervals left out
as a variant.

### Tables used by the claims

`cells` has one row per cell (a putative excitatory unit with contamination at most 0.5 and at least 100
spikes in session 1) and recording; `recs` has one row per recording. H4825 is left out. The archive
grouping variant uses the archive's genotype, and the archive's age group where the birth date gives one
(H3029 is then adult and its cells are selected with the adult threshold).

In [ ]:
cell_rows, rec_rows, arch_rows = [], [], []
first_day = meta[~meta.subject.isin(hdfx.DUPLICATE_OF)].sort_values("session_id").groupby("subject").session_id.first()
for r in results:
    m = r["meta"]
    if r["main"] is None:
        continue
    base = {"subject": m["subject"], "session": m["session_id"], "age_group": m["age_group"], "genotype": m["genotype"],
            "first_day": first_day[m["subject"]] == m["session_id"]}
    rec_rows.append({**base, **r["main"]["rec"]})
    c = r["main"]["cells"].copy()
    for k, v in base.items():
        c[k] = v
    cell_rows.append(c)
    a_res = r["other_age"] if r["other_age"] is not None else r["main"]
    ca = a_res["cells"].copy()
    for k, v in base.items():
        ca[k] = v
    ca["genotype"] = m["archive_genotype"]
    ca["age_group"] = m["archive_age_group"] or m["age_group"]
    arch_rows.append(ca)
cells = pd.concat(cell_rows, ignore_index=True)
cells_arch = pd.concat(arch_rows, ignore_index=True)
recs = pd.DataFrame(rec_rows)
cells["animal"] = cells.subject
cells_arch["animal"] = cells_arch.subject
print("cells per group (main):")
print(cells.groupby(["age_group", "genotype"]).agg(rats=("subject", "nunique"), recordings=("session", "nunique"), cells=("unit", "size")).to_string())
print("\ncells per group (archive labels):")
print(cells_arch.groupby(["age_group", "genotype"]).agg(rats=("subject", "nunique"), recordings=("session", "nunique"), cells=("unit", "size")).to_string())
print("\nper recording:")
print(recs[["subject", "session", "age_group", "genotype", "n_units", "n_exc_ttp", "n_exc", "n_cells", "n_sessions", "rem_s", "n_cells_rem", "rem_in_sessions", "rem_in_sessions_s"]].to_string())
print("\nranges of the cell measures (main):")
print(cells.describe().T[["count", "min", "50%", "max"]].round(3).to_string())

In [ ]:
def lme_variants(value, age, higher="KO", log_ok=True, uniform_col=None, extra=None):
    '''The LME test and its variants for one cell measure in one age group.'''
    d = cells[cells.age_group == age]
    rows = []
    def add(name, df, col=value, log=False):
        coef, p, n = hdfx.lme_lrt(df, col, log=log)
        rows.append({"test": name, "n": n, "coef_KO_minus_WT": coef, "p": p})
    add("LME (main)", d)
    if log_ok:
        add("LME on log values", d, log=True)
    if not d.first_day.all():
        add("LME, first day of each rat", d[d.first_day])
    add("LME, archive labels", cells_arch[cells_arch.age_group == age])
    if age == "juvenile":
        add("LME, H3026 left out", d[d.subject != "H3026"])
    if uniform_col:
        add("LME, uniform occupancy (v2)", d, col=uniform_col)
    if extra is not None:
        add(*extra)
    ko, wt = d.loc[d.genotype == "KO", value].dropna(), d.loc[d.genotype == "WT", value].dropna()
    mw = stats.mannwhitneyu(ko, wt)
    rows.append({"test": "Mann-Whitney on cells", "n": len(ko) + len(wt), "coef_KO_minus_WT": ko.median() - wt.median(), "p": mw.pvalue})
    am = d.groupby(["animal", "genotype"])[value].mean().reset_index()
    tt = stats.ttest_ind(am.loc[am.genotype == "KO", value], am.loc[am.genotype == "WT", value])
    rows.append({"test": f"t-test on animal means, t({len(am) - 2}) = {tt.statistic:.2f}", "n": len(am),
                 "coef_KO_minus_WT": am.loc[am.genotype == "KO", value].mean() - am.loc[am.genotype == "WT", value].mean(), "p": tt.pvalue})
    out = pd.DataFrame(rows)
    summ = d.groupby("genotype")[value].agg(["count", "median", "mean", "min", "max"])
    per_animal = d.groupby(["genotype", "animal"])[value].agg(["count", "median", "mean"])
    return out, summ, per_animal


def show(value, age, **kw):
    out, summ, per_animal = lme_variants(value, age, **kw)
    print(f"{value}, {age}: cells by genotype")
    print(summ.round(3).to_string())
    print("\nper rat")
    print(per_animal.round(3).to_string())
    print("\ntests")
    print(out.round(4).to_string())
    return out


def corr(age, x, y, df=None):
    d = (cells if df is None else df)
    d = d[d.age_group == age].dropna(subset=[x, y])
    rows = []
    for g in ("WT", "KO"):
        e = d[d.genotype == g]
        r, p = stats.pearsonr(e[x], e[y])
        f = e[e.first_day]
        rows.append({"genotype": g, "cells": len(e), "rats": e.animal.nunique(), "r": r, "p": p,
                     "spearman": stats.spearmanr(e[x], e[y])[0],
                     "r_first_day": stats.pearsonr(f[x], f[y])[0], "cells_first_day": len(f),
                     "x_range": (round(e[x].min(), 3), round(e[x].max(), 3)), "y_range": (round(e[y].min(), 3), round(e[y].max(), 3))})
    return pd.DataFrame(rows)


def scatter(age, x, y, xlabel, ylabel):
    """Per-cell scatter of y against x by genotype, cells of later days marked."""
    d = cells[cells.age_group == age].dropna(subset=[x, y])
    fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
    for k, g in enumerate(("WT", "KO")):
        e = d[d.genotype == g]
        for first, mk, lab in ((True, "o", "first day"), (False, "x", "later day")):
            f = e[e.first_day == first]
            ax[k].scatter(f[x], f[y], s=8, marker=mk, alpha=0.6, label=lab)
        hi = max(d[x].max(), d[y].max())
        ax[k].plot([0, hi], [0, hi], c="0.6", lw=0.8)
        ax[k].set_title(f"{age} {g}")
        ax[k].set_xlabel(xlabel)
    ax[0].set_ylabel(ylabel)
    ax[0].legend()
    plt.tight_layout()
    plt.show()
    print(f"plotted ranges: {x} {d[x].min():.3f} to {d[x].max():.3f}; {y} {d[y].min():.3f} to {d[y].max():.3f} (no point outside the axes)")
    e = d.assign(diff=(d[y] - d[x]).abs()).sort_values("diff", ascending=False)
    print("the ten cells farthest from the diagonal:")
    print(e[["subject", "session", "genotype", "unit", "first_day", x, y, "diff"]].head(10).round(3).to_string())

## unit-counts

> 406 units were recorded from 7 juvenile WT rats and 372 from 5 juvenile Fmr1-/y rats; 543 units from
> 6 adult WT rats and 507 from 6 adult Fmr1-/y rats.

In [ ]:
expected = {("juvenile", "WT"): (406, 7), ("juvenile", "KO"): (372, 5), ("adult", "WT"): (543, 6), ("adult", "KO"): (507, 6)}
exc_units = pd.Series({(r["meta"]["subject"], r["meta"]["session_id"]): r["main"]["rec"]["n_exc_ttp"] for r in results if r["main"] is not None})
def table(df, age_col, geno_col, reading):
    rows = []
    for (age, geno), (eu, er) in expected.items():
        d = df[(df[age_col] == age) & (df[geno_col] == geno)]
        n = d.n_units.sum() if reading == "a" else sum(exc_units.get((s, ss), 0) for s, ss in zip(d.subject, d.session_id))
        rows.append({"group": f"{age} {geno}", "units": n, "paper_units": eu, "units_ok": abs(n - eu) <= 0.1 * eu,
                     "rats": d.subject.nunique(), "paper_rats": er, "rats_ok": abs(d.subject.nunique() - er) <= 0.1 * er})
    t = pd.DataFrame(rows)
    t["group_ok"] = t.units_ok & t.rats_ok
    return t
for reading in ("a", "b"):
    t = table(once, "age_group", "genotype", reading)
    print(f"main grouping, reading ({reading}) {'all units' if reading == 'a' else 'putative excitatory units'}:")
    print(t.to_string(), "\ngroups matching:", int(t.group_ok.sum()), "of 4\n")
print("variant, archive labels (all units, each file once):")
print(table(arch[~arch.subject.isin(hdfx.DUPLICATE_OF)], "archive_age_group", "archive_genotype", "a").to_string())
print("\nvariant, main grouping counting H4825's file as a recording of its own (all units):")
print(table(meta, "age_group", "genotype", "a").to_string())

**Result: partially reproduced.** Counting every unit of each recording once, the juvenile WT (406
units, 7 rats), juvenile Fmr1-/y (372, 5) and adult WT (543, 6) groups match the paper exactly, with
H3026 taken as Fmr1-/y and H3029 as a juvenile. The adult Fmr1-/y group has 440 units in 5 rats against
507 in 6, because H4825's file holds H4824's recording. Counting H4825's file as a recording of its own
gives 528 units in 6 rats, within the tolerance, but it counts one recording twice. As putative
excitatory units no group matches, and with the archive's labels no group matches.

## units-per-recording

> Juvenile recordings held 8 to 83 putative excitatory units each, adult recordings 30 to 114.

In [ ]:
rr = recs.copy()
rows = []
for reading, col in (("a all units", "n_units"), ("b putative excitatory (trough-to-peak)", "n_exc_ttp"), ("c cells of session 1", "n_cells")):
    for age, (lo, hi) in (("juvenile", (8, 83)), ("adult", (30, 114))):
        d = rr[rr.age_group == age][col]
        rows.append({"reading": reading, "age": age, "min": d.min(), "max": d.max(), "paper": f"{lo} to {hi}",
                     "min_ok": abs(d.min() - lo) <= 0.1 * lo, "max_ok": abs(d.max() - hi) <= 0.1 * hi})
print(pd.DataFrame(rows).to_string())
print("\nadult recordings sorted by units:", rr[rr.age_group == "adult"].sort_values("n_units")[["subject", "n_units", "n_exc_ttp", "n_cells"]].values.tolist())

**Result: partially reproduced.** Counting all units, the juvenile recordings hold 8 to 83, as the
paper says, and the adult ones 55 to 114, so the adult minimum differs. As putative excitatory units by
trough-to-peak duration, the ranges are 5 to 50 (juvenile) and 30 to 81 (adult). The paper's juvenile
range and adult maximum are counts of all units, and its adult minimum is the count of putative
excitatory units in H4822, so the reported ranges mix the two readings.

## figs1c-juvenile-behaviour-similar and figs2c-adult-behaviour-similar

> Juvenile (and adult) Fmr1-/y and WT rats do not differ in the time spent inactive, the time spent in
> the centre of the arena, or the speed while active.

In [ ]:
def behaviour(age):
    d = recs[recs.age_group == age].groupby(["subject", "genotype"])[["inactive", "centre", "active_speed"]].mean().reset_index()
    rows = []
    for col in ("inactive", "centre", "active_speed"):
        ko, wt = d.loc[d.genotype == "KO", col], d.loc[d.genotype == "WT", col]
        t = stats.ttest_ind(ko, wt)
        rows.append({"measure": col, "WT_mean": wt.mean(), "KO_mean": ko.mean(), "WT_range": (round(wt.min(), 3), round(wt.max(), 3)),
                     "KO_range": (round(ko.min(), 3), round(ko.max(), 3)), "df": len(d) - 2, "t": t.statistic, "p": t.pvalue})
    return d, pd.DataFrame(rows)
for age in ("juvenile", "adult"):
    d, t = behaviour(age)
    print(f"{age}: per rat (fractions of tracked time; speed in cm/s)")
    print(d.round(3).to_string())
    print(t.round(4).to_string(), "\n")
print("variants of the thresholds (rat means): inactive and active at 2 cm/s; centre disc of radius one third of the diameter")
for age in ("juvenile", "adult"):
    d = recs[recs.age_group == age].groupby(["subject", "genotype"])[["inactive_2", "centre_third", "active_speed_2"]].mean().reset_index()
    for col in ("inactive_2", "centre_third", "active_speed_2"):
        tt = stats.ttest_ind(d.loc[d.genotype == "KO", col], d.loc[d.genotype == "WT", col])
        print(f"  {age} {col}: WT {d.loc[d.genotype == 'WT', col].mean():.3f}, KO {d.loc[d.genotype == 'KO', col].mean():.3f}, t({len(d) - 2}) = {tt.statistic:.2f}, p = {tt.pvalue:.4f}")
print("variants: first day of each rat only")
for age in ("juvenile", "adult"):
    keep = recs.first_day
    d = recs[keep & (recs.age_group == age)]
    for col in ("inactive", "centre", "active_speed"):
        tt = stats.ttest_ind(d.loc[d.genotype == "KO", col], d.loc[d.genotype == "WT", col])
        print(f"  {age} {col}: t({len(d) - 2}) = {tt.statistic:.2f}, p = {tt.pvalue:.4f}")

**Result: figs1c-juvenile-behaviour-similar reproduced; figs2c-adult-behaviour-similar reproduced.**
In juveniles no measure differs by t-test on rat means at 0.05: time inactive p = 0.39, time in the
centre p = 0.093, speed while active p = 0.063 (Fmr1-/y 5.74 against WT 4.88 cm/s). The active speed
differs with the first day of each rat alone (p = 0.034), so the absence of a speed difference is
fragile; with thresholds of 2 cm/s and a centre disc of a third of the diameter, p = 0.25, 0.11 and
0.26. In adults none differs (p = 0.91, 0.355 and 0.70).

## figs1-juvenile-cell-metrics-similar and figs2-adult-cell-metrics-similar

> Cluster contamination, mean and peak firing rates, and resultant vector length of PoSub cells do not
> differ between genotypes (juveniles, Figure S1f-i; adults, Figure S2f-i).

In [ ]:
metric_tests = {}
for age in ("juvenile", "adult"):
    for col in ("contamination", "mean_rate", "peak_rate", "vector_length"):
        print("=" * 100)
        metric_tests[(age, col)] = show(col, age, log_ok=col != "contamination")
print("\nmain LME p per measure:")
print(pd.Series({k: v.loc[0, "p"] for k, v in metric_tests.items()}).round(4).to_string())

**Result: figs1-juvenile-cell-metrics-similar reproduced; figs2-adult-cell-metrics-similar
reproduced.** No main LME test finds a genotype effect on contamination, mean rate, peak rate or vector
length (juveniles p = 0.11 to 0.72; adults p = 0.10 to 0.34). Mann-Whitney tests on cells, which ignore
the rat, find differences in juvenile mean rate (p = 0.038) and vector length (p = 0.0001), and in adult
mean rate, peak rate and vector length (p = 0.022, 0.0026 and 0.0032).

## fig1ef-juvenile-sharper-tuning

> PoSub cells of juvenile Fmr1-/y rats are more sharply tuned to head direction than those of WT
> littermates, with higher HD information and narrower tuning curves.

In [ ]:
t_info = show("hd_info", "juvenile", uniform_col="hd_info_uniform")
print("=" * 100)
t_hw = show("half_width", "juvenile")
d = cells[cells.age_group == "juvenile"]
print("\nfraction of cells with half-width below 120 degrees:", d.groupby("genotype").half_width.apply(lambda x: np.mean(x < 120)).round(3).to_dict())
print("cells with half-width 360 (never below half maximum):", d.groupby("genotype").half_width.apply(lambda x: int(np.sum(x >= 360))).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for i, (col, lab) in enumerate((("hd_info", "HD info (bits/spike)"), ("half_width", "half-width (deg)"))):
    for g, c in (("WT", "0.5"), ("KO", "C3")):
        ax[i].hist(d.loc[d.genotype == g, col], 30, histtype="step", color=c, density=True, label=g)
    ax[i].set_xlabel(lab)
ax[0].legend()
plt.tight_layout()
plt.show()
print("plotted ranges: HD info", d.hd_info.min().round(3), "to", d.hd_info.max().round(3), "; half-width", d.half_width.min(), "to", d.half_width.max())

**Result: partially reproduced.** HD information is higher in juvenile Fmr1-/y cells than in WT
cells (median 0.596 against 0.189 bits/spike; LME p = 0.030), and stays so with the first day of each
rat alone (p = 0.035), but not on log values (p = 0.12), with uniform occupancy as in version 2
(p = 0.081), on rat means (t(10) = 2.11, p = 0.061), with H3026 left out (p = 0.072) or with the
archive's labels (p = 0.23). The half-width does not differ (median 50 against 62 degrees; LME
p = 0.54).

## fig1g-juvenile-halves-correlation

> In juveniles the HD information of a cell is stable between the two halves of an exploration session
> (WT r = 0.90, Fmr1-/y r = 0.95).

In [ ]:
print(corr("juvenile", "info_half1", "info_half2").round(4).to_string())
print("\nvariant, archive labels:")
print(corr("juvenile", "info_half1", "info_half2", cells_arch).round(4).to_string())
print("\nvariant, H3026 left out:")
print(corr("juvenile", "info_half1", "info_half2", cells[cells.subject != "H3026"]).round(4).to_string())
scatter("juvenile", "info_half1", "info_half2", "HD info, first half (bits/spike)", "HD info, second half")

**Result: partially reproduced.** HD information of the two halves of a session correlates with
r = 0.815 over 323 WT cells, within the tolerance of the paper's 0.90 (lower bound 0.81), and r = 0.835
over 237 Fmr1-/y cells, below the paper's 0.95 by more than the tolerance (lower bound 0.855). The WT
value is 0.005 above its bound, and with the first day of each rat alone it is 0.739. Spearman
correlations are higher (0.863 and 0.909). Five of the ten cells farthest from the diagonal are from
H3026, the rat counted as Fmr1-/y against the archive's label; with H3026 left out the Fmr1-/y r is
0.869, within tolerance, and with the first day alone it is 0.898.

## fig1h-juvenile-smaller-offset

> Within a session, the tuning curves of juvenile Fmr1-/y cells drift less relative to the room than
> those of WT cells; the angular offset between the two halves is smaller.

In [ ]:
d = cells[cells.age_group == "juvenile"]
t_off = show("offset_halves", "juvenile", log_ok=False,
             extra=("LME, cells with half-width below 120 deg", d[d.half_width < 120]))
print("plotted quantity range (deg):", d.offset_halves.min(), "to", d.offset_halves.max())

**Result: reproduced.** The offset between the two halves' tuning curves is smaller in juvenile
Fmr1-/y cells (median 7 against 26 degrees; LME p = 0.035). It holds with the first day alone
(p = 0.017), and not for cells with half-width below 120 degrees (p = 0.060), on rat means (t(10) =
-2.09, p = 0.063), with H3026 left out (p = 0.068) or with the archive's labels (p = 0.17).

## fig1j-juvenile-less-drift

> Bayesian decoding shows less drift of the HD signal relative to the room in juvenile Fmr1-/y rats than
> in WT rats.

In [ ]:
MIN_DRIFT_BINS = 300   # 1 min of decoded movement; a choice, shown as a variant


def drift_test(age, col):
    d = recs[(recs.age_group == age)].dropna(subset=[col])
    am = d.groupby(["subject", "genotype"])[col].mean().reset_index()
    ko, wt = am.loc[am.genotype == "KO", col], am.loc[am.genotype == "WT", col]
    t = stats.ttest_ind(ko, wt)
    return am, {"measure": col, "WT_mean": wt.mean(), "KO_mean": ko.mean(), "rats": f"{len(wt)} WT, {len(ko)} KO",
                "t": t.statistic, "df": len(am) - 2, "p": t.pvalue}
def drift_report(age):
    d = recs[recs.age_group == age]
    print("recordings without decoding (fewer than 6 cells):", d.loc[d.drift_mean.isna(), ["subject", "session", "n_cells"]].values.tolist())
    print("second-half bins of 200 ms: all, with valid tracking, and valid with speed above 2 cm/s (decoded and averaged):")
    print(d[["subject", "session", "genotype", "n_cells", "drift_bins_all", "drift_bins_valid", "drift_bins", "drift_mean", "drift_sd", "drift_mean_v2"]].round(2).to_string())
    rows = []
    for col in ("drift_mean", "drift_sd", "drift_mean_v2", "drift_sd_v2"):
        am, r = drift_test(age, col)
        rows.append(r)
    print("\nper rat (mean over recordings):")
    print(drift_test(age, "drift_mean")[0].round(2).to_string())
    print("\nt-tests on rat means (degrees):")
    print(pd.DataFrame(rows).round(4).to_string())
    first = recs[(recs.age_group == age) & recs.first_day].dropna(subset=["drift_mean"])
    ko, wt = first.loc[first.genotype == "KO", "drift_mean"], first.loc[first.genotype == "WT", "drift_mean"]
    t = stats.ttest_ind(ko, wt)
    print(f"variant, first day only: WT {wt.mean():.2f}, KO {ko.mean():.2f}, t({len(first) - 2}) = {t.statistic:.2f}, p = {t.pvalue:.4f}")
    arch_g = meta.set_index(["subject", "session_id"]).archive_genotype
    dd = d.dropna(subset=["drift_mean"]).copy()
    dd["genotype"] = [arch_g[(s, ss)] for s, ss in zip(dd.subject, dd.session)]
    am = dd.groupby(["subject", "genotype"]).drift_mean.mean().reset_index()
    t = stats.ttest_ind(am.loc[am.genotype == "KO", "drift_mean"], am.loc[am.genotype == "WT", "drift_mean"])
    print(f"variant, archive genotypes: t({len(am) - 2}) = {t.statistic:.2f}, p = {t.pvalue:.4f}")
    for name, sub in ((f"recordings with at least {MIN_DRIFT_BINS} averaged bins", d[d.drift_bins >= MIN_DRIFT_BINS]),
                      ("H3026 left out", d[d.subject != "H3026"])):
        if age == "adult" and name.startswith("H3026"):
            continue
        sub = sub.dropna(subset=["drift_mean"])
        am = sub.groupby(["subject", "genotype"]).drift_mean.mean().reset_index()
        ko, wt = am.loc[am.genotype == "KO", "drift_mean"], am.loc[am.genotype == "WT", "drift_mean"]
        t = stats.ttest_ind(ko, wt)
        print(f"variant, {name}: {len(wt)} WT, {len(ko)} KO rats; WT {wt.mean():.2f}, KO {ko.mean():.2f}, t({len(am) - 2}) = {t.statistic:.2f}, p = {t.pvalue:.4f}")
drift_report("juvenile")

**Result: not reproduced.** Decoder drift is lower in Fmr1-/y rats (rat means 17.0 against 35.0
degrees) but the t-test on rat means is not significant (t(10) = -1.91, p = 0.085). With version 2's
1.5 cm/s threshold p = 0.064, for the SD of drift p = 0.12, with the first day alone p = 0.088, with
the archive's labels p = 0.20, with H3026 left out p = 0.14. H3019 (210618-1) is left out with 5
cells. H3001 (200201) is decoded on 279 bins, where the other recordings have 491 to 2480, as the rat
moved faster than 2 cm/s in few bins of the second half; leaving out recordings under 300 bins gives
p = 0.085.

## figs1j-juvenile-epochs-correlation and figs1k-juvenile-epoch-drift-similar

> In juveniles HD information is preserved between the two exploration sessions (WT r = 0.85, Fmr1-/y
> r = 0.85), and the drift of tuning curves between the sessions does not differ between genotypes.

In [ ]:
print(corr("juvenile", "hd_info", "info_s2").round(4).to_string())
scatter("juvenile", "hd_info", "info_s2", "HD info, session 1 (bits/spike)", "HD info, session 2")
print()
t_ep = show("offset_epochs", "juvenile", log_ok=False)

**Result: figs1j-juvenile-epochs-correlation partially reproduced; figs1k-juvenile-epoch-drift-similar
reproduced.** HD information correlates between the two sessions with r = 0.853 over 309 WT cells
(paper 0.85) and 0.753 over 225 Fmr1-/y cells (paper 0.85, tolerance down to 0.765), so one of the two
values is within tolerance. The offset between sessions does not differ between genotypes (median 14
against 21 degrees; LME p = 0.87; first day p = 0.72; archive labels p = 0.44).

## fig2c-juvenile-isomap-wake-correlation and fig2d-juvenile-isomap-wake-sharper

> In juveniles the HD information of tuning curves computed against the Isomap angle of waking
> population activity matches that of real HD tuning curves (WT r = 0.84, Fmr1-/y r = 0.92), and
> juvenile Fmr1-/y cells are better tuned than WT cells to this internal HD signal.

In [ ]:
def iso_report(age):
    d = recs[recs.age_group == age]
    print("Isomap alignment with real HD (Rayleigh length of the error) per recording; recordings with fewer than 6 cells left out:")
    print(d[["subject", "session", "genotype", "n_cells", "iso_alignment", "iso_flipped"]].round(3).to_string())
    print(f"alignment range {d.iso_alignment.min():.3f} to {d.iso_alignment.max():.3f}")
    c = cells[cells.age_group == age]
    for col in ("iso_wake_info", "real200_info"):
        top = c.sort_values(col, ascending=False).groupby("genotype").head(3)
        print(f"three highest {col} per genotype:")
        print(top[["genotype", "subject", "session", "unit", "n_spikes_s1", "iso_wake_info", "real200_info", "hd_info"]].round(4).to_string())
iso_report("juvenile")
print()
print(corr("juvenile", "real200_info", "iso_wake_info").round(4).to_string())
print()
t_iso = show("iso_wake_info", "juvenile")

In [ ]:
# Isomap fitted on at most 3000 bins (as in the first round of this attempt), beside the fit on every bin used in
# the analysis, for the recording with the most cells in each age group
seed_of = {(m["subject"], m["session_id"]): k for k, m in enumerate(meta.to_dict("records"))}
fit_rows = []
for age in ("juvenile", "adult"):
    rr_ = recs[recs.age_group == age].sort_values("n_cells").iloc[-1]
    k = seed_of[(rr_.subject, rr_.session)]
    s_ = hdfx.load_session(assets.asset_id[k])
    sub_ = hdfx.analyse(s_, age, seed=k, fit_max=3000)["cells"].set_index("unit")
    full_ = cells[(cells.subject == rr_.subject) & (cells.session == rr_.session)].set_index("unit")
    for col in ("iso_wake_info", "iso_rem_info"):
        b_ = full_[col].dropna()
        a_ = sub_[col].reindex(b_.index)
        fit_rows.append({"recording": f"{rr_.subject} {rr_.session}", "age": age, "measure": col, "cells": len(b_),
                         "median_full": b_.median(), "median_3000_bins": a_.median(),
                         "r_full_vs_3000": stats.pearsonr(a_, b_)[0],
                         "r_with_real200_full": stats.pearsonr(full_.loc[b_.index, "real200_info"], b_)[0] if col == "iso_wake_info" else np.nan,
                         "r_with_real200_3000": stats.pearsonr(full_.loc[b_.index, "real200_info"], a_)[0] if col == "iso_wake_info" else np.nan})
    del s_
print(pd.DataFrame(fit_rows).round(3).to_string())

**Result: fig2c-juvenile-isomap-wake-correlation reproduced; fig2d-juvenile-isomap-wake-sharper
not reproduced.** The HD information of Isomap tuning curves correlates with that of real-HD curves
with r = 0.815 in WT and 0.910 in Fmr1-/y (paper 0.84 and 0.92). Isomap HD information is higher in
Fmr1-/y cells (median 0.416 against 0.243 bits/spike) but the LME test is not significant (p = 0.056);
no variant that keeps the rat is below 0.05 (log values p = 0.13, first day p = 0.15, archive labels
p = 0.14, H3026 left out p = 0.075, rat means p = 0.089), and only the Mann-Whitney test on cells is
(p = 0.0007). The direction matches the paper. The Isomap angle aligned with real HD with Rayleigh
lengths of 0.21 to 0.90. Isomap is fitted on every bin; fitting it on 3000 bins, as in the first round of this
attempt, raises the median Isomap information of H3026 211003 from 0.435 to 0.546 bits/spike (r = 0.987
across its cells), while in H4823 the two fits agree (0.360 and 0.359).

## fig2g-juvenile-isomap-rem-correlation and fig2h-juvenile-isomap-rem-sharper

> In juveniles the HD information of tuning curves computed against the Isomap angle of REM activity
> matches that of real HD tuning curves (WT r = 0.73, Fmr1-/y r = 0.90), and juvenile Fmr1-/y cells are
> better tuned than WT cells to the internally generated signal during REM.

In [ ]:
d = recs[recs.age_group == "juvenile"]
print(d[["subject", "session", "genotype", "rem_s", "n_cells_rem", "rem_bins", "rem_in_sessions", "rem_in_sessions_s"]].to_string())
print()
print(corr("juvenile", "real200_info", "iso_rem_info").round(4).to_string())
print()
t_rem = show("iso_rem_info", "juvenile",
             extra=("LME, REM inside exploration left out", cells[cells.age_group == "juvenile"], "iso_rem_info_outside"))
print("\nvariant, REM inside exploration left out:")
print(corr("juvenile", "real200_info", "iso_rem_info_outside").round(4).to_string())

**Result: fig2g-juvenile-isomap-rem-correlation partially reproduced; fig2h-juvenile-isomap-rem-sharper
reproduced.** The HD information of REM Isomap tuning curves correlates with that of waking real-HD
curves with r = 0.833 in Fmr1-/y (paper 0.90, within tolerance) and 0.831 in WT (paper 0.73, above the
tolerance of 0.657 to 0.803). REM Isomap HD information is higher in Fmr1-/y cells (median 0.401
against 0.169 bits/spike; LME p = 0.045). The result is at the threshold: on log values p = 0.092,
with the first day alone p = 0.16, with the archive's labels p = 0.25, with H3026 left out p = 0.095,
on rat means p = 0.079; leaving out the REM intervals inside exploration sessions gives p = 0.045.

## fig3ef-adult-tuning-similar

> In adults HD tuning does not differ between Fmr1-/y and WT rats, in HD information or tuning curve
> width.

In [ ]:
a_info = show("hd_info", "adult", uniform_col="hd_info_uniform")
print("=" * 100)
a_hw = show("half_width", "adult")

**Result: reproduced.** In adults neither HD information (median 0.193 in Fmr1-/y against 0.318
in WT; LME p = 0.55) nor half-width (78 against 76 degrees; p = 0.59) differs between genotypes. The
Mann-Whitney test on cells finds lower HD information in Fmr1-/y cells (p = 0.0095).

## fig3g-adult-halves-correlation

> In adults HD information is stable between the two halves of an exploration session (WT r = 0.94,
> Fmr1-/y r = 0.94).

In [ ]:
print(corr("adult", "info_half1", "info_half2").round(4).to_string())
scatter("adult", "info_half1", "info_half2", "HD info, first half (bits/spike)", "HD info, second half")

**Result: not reproduced.** In adults HD information correlates between the two halves of a
session with r = 0.804 over 395 WT cells and 0.748 over 282 Fmr1-/y cells, both below the lower bound
of 0.846 around the paper's 0.94. Spearman correlations are 0.862 and 0.818. The cells farthest from the
diagonal have higher information in the second half and come from several rats of both genotypes.

## fig3h-adult-larger-offset

> Within a session the tuning curves of adult Fmr1-/y cells drift more relative to the room than those
> of WT cells.

In [ ]:
d = cells[cells.age_group == "adult"]
a_off = show("offset_halves", "adult", log_ok=False,
             extra=("LME, cells with half-width below 120 deg", d[d.half_width < 120]))

**Result: reproduced.** The offset between the halves is larger in adult Fmr1-/y cells (median 60
against 20 degrees; LME p = 0.030), with the archive's labels (p = 0.043) and for cells with half-width
below 120 degrees (p = 0.017); on rat means it is not significant (t(9) = 2.20, p = 0.056).

## fig3j-adult-more-drift

> Bayesian decoding shows more drift of the HD signal relative to the room in adult Fmr1-/y rats than in
> WT rats.

In [ ]:
drift_report("adult")

**Result: reproduced.** Decoder drift is higher in adult Fmr1-/y rats (rat means 68.4 against 20.1
degrees; t(9) = 2.93, p = 0.017; with version 2's threshold p = 0.016; for the SD of drift p = 0.099),
over 5 Fmr1-/y rats where the paper has 6. Three Fmr1-/y rats drift by 91 to 101 degrees, and two
(H4819 at 32.9, H4827 at 18.3) lie within the WT range of 11.3 to 37.1.

## figs2j-adult-epochs-correlation and figs2k-adult-epoch-drift-larger

> In adults HD information is preserved between the two exploration sessions (WT r = 0.90, Fmr1-/y
> r = 0.76), and Fmr1-/y tuning curves drift more than WT ones between the sessions.

In [ ]:
print(corr("adult", "hd_info", "info_s2").round(4).to_string())
scatter("adult", "hd_info", "info_s2", "HD info, session 1 (bits/spike)", "HD info, session 2")
print()
a_ep = show("offset_epochs", "adult", log_ok=False)

**Result: figs2j-adult-epochs-correlation not reproduced; figs2k-adult-epoch-drift-larger
reproduced.** HD information correlates between the sessions with r = 0.787 over 395 WT cells and 0.562
over 276 Fmr1-/y cells, both below the lower bounds (0.81 and 0.684), though lower in Fmr1-/y as in the
paper (0.90 and 0.76). Spearman correlations are 0.888 and 0.714; five of the ten cells farthest from
the diagonal are from H4817, whose cells gain information in session 2. The offset between the
sessions is larger in Fmr1-/y cells (median 74 against 30 degrees; LME p = 0.028; archive labels
p = 0.022; rat means p = 0.055).

## fig4c-adult-isomap-wake-correlation and fig4d-adult-isomap-wake-similar

> In adults the HD information of Isomap tuning curves from waking activity matches that of real HD
> tuning curves (WT r = 0.95, Fmr1-/y r = 0.90), and does not differ between genotypes.

In [ ]:
iso_report("adult")
print()
print(corr("adult", "real200_info", "iso_wake_info").round(4).to_string())
print()
a_iso = show("iso_wake_info", "adult")

**Result: fig4c-adult-isomap-wake-correlation partially reproduced; fig4d-adult-isomap-wake-similar
reproduced.** Isomap and real-HD information correlate with r = 0.945 in WT (paper 0.95, within
tolerance) and 0.614 in Fmr1-/y (paper 0.90, outside). Isomap HD information does not differ between
genotypes (median 0.279 in Fmr1-/y against 0.336 in WT; LME p = 0.79; rat means p = 0.80). The Isomap
angle aligned poorly with real HD in H4822 (0.152), H4824 (0.224) and H4817 (0.386), the three
Fmr1-/y rats with the largest decoder drift; the three Fmr1-/y cells with the highest Isomap
information are from H4817 and carry little real-HD information (0.45 to 0.67 bits/spike).

## fig4g-adult-isomap-rem-correlation and fig4h-adult-isomap-rem-similar

> In adults the HD information of Isomap tuning curves from REM activity matches that of real HD tuning
> curves (WT r = 0.82, Fmr1-/y r = 0.80), and does not differ between genotypes.

In [ ]:
d = recs[recs.age_group == "adult"]
print(d[["subject", "session", "genotype", "rem_s", "n_cells_rem", "rem_bins", "rem_in_sessions", "rem_in_sessions_s"]].to_string())
print()
print(corr("adult", "real200_info", "iso_rem_info").round(4).to_string())
print()
a_rem = show("iso_rem_info", "adult",
             extra=("LME, REM inside exploration left out", cells[cells.age_group == "adult"], "iso_rem_info_outside"))
print("\nvariant, REM inside exploration left out:")
print(corr("adult", "real200_info", "iso_rem_info_outside").round(4).to_string())
print(f"\nrun time so far {time.time() - started:.0f} s")

**Result: fig4g-adult-isomap-rem-correlation partially reproduced; fig4h-adult-isomap-rem-similar
reproduced.** REM Isomap and waking real-HD information correlate with r = 0.820 over 378 WT cells
(paper 0.82) and 0.651 over 280 Fmr1-/y cells (paper 0.80, below the bound of 0.72; 0.691 with the REM
inside exploration left out). REM Isomap HD information does not differ between genotypes (median
0.413 in Fmr1-/y against 0.332 in WT; LME p = 0.27; with the REM inside exploration left out
p = 0.73).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| unit-counts | partially reproduced (3 of 4 groups exact; adult Fmr1-/y 440 units in 5 rats) |
| units-per-recording | partially reproduced (juvenile 8 to 83; adult 55 to 114 for all units) |
| figs1c-juvenile-behaviour-similar | reproduced (active speed p = 0.063; first day p = 0.034) |
| figs2c-adult-behaviour-similar | reproduced |
| figs1-juvenile-cell-metrics-similar | reproduced |
| figs2-adult-cell-metrics-similar | reproduced |
| fig1ef-juvenile-sharper-tuning | partially reproduced (HD information p = 0.030, half-width p = 0.54) |
| fig1g-juvenile-halves-correlation | partially reproduced (WT r 0.815 within, Fmr1-/y 0.835 below 0.855) |
| fig1h-juvenile-smaller-offset | reproduced (p = 0.035) |
| fig1j-juvenile-less-drift | not reproduced (p = 0.085, same direction) |
| figs1j-juvenile-epochs-correlation | partially reproduced |
| figs1k-juvenile-epoch-drift-similar | reproduced |
| fig2c-juvenile-isomap-wake-correlation | reproduced |
| fig2d-juvenile-isomap-wake-sharper | not reproduced (p = 0.056, same direction) |
| fig2g-juvenile-isomap-rem-correlation | partially reproduced |
| fig2h-juvenile-isomap-rem-sharper | reproduced (p = 0.045) |
| fig3ef-adult-tuning-similar | reproduced |
| fig3g-adult-halves-correlation | not reproduced (r 0.804 and 0.748 against 0.94) |
| fig3h-adult-larger-offset | reproduced (p = 0.030) |
| fig3j-adult-more-drift | reproduced |
| figs2j-adult-epochs-correlation | not reproduced (r 0.787 and 0.562 against 0.90 and 0.76) |
| figs2k-adult-epoch-drift-larger | reproduced |
| fig4c-adult-isomap-wake-correlation | partially reproduced |
| fig4d-adult-isomap-wake-similar | reproduced |
| fig4g-adult-isomap-rem-correlation | partially reproduced |
| fig4h-adult-isomap-rem-similar | reproduced |
| figs1d-probe-in-posub | blocked (histology without annotation) |

Several juvenile results sit near the 0.05 threshold and depend on the grouping. HD information
(fig1ef, p = 0.030), the offset between halves (fig1h, p = 0.035) and REM Isomap information (fig2h,
p = 0.045) lose significance with the archive's labels and with H3026 left out; fig1ef and fig1h hold
with the first day of each rat alone, fig2h does not. Waking (fig2d) and REM (fig2h) Isomap information
fall on opposite sides of 0.05 (p = 0.056 and 0.045) with LME coefficients of 0.196 and 0.207.

**How the analysis differs from the paper's.** The genotype of H3026 and the age group of H3029 are
taken from the paper's unit counts, not from the archive; the archive's labels and the juvenile
results with H3026 left out are shown as variants. H4825's file is a re-timed copy of H4824's
recording, so the adult Fmr1-/y group has 5 rats where the paper has 6. Cells re-recorded on
consecutive days could not be excluded (the files give no probe depth per day), so they are counted
on each day; the first day of each rat is shown as a variant. Exploration sessions are taken from the
tracking, as the epoch tables do not mark them reliably, and cut to 20 min. The LME is a
random-intercept linear model on the raw values; the paper chose between linear, lognormal and gamma
models by the data. The decoder follows version 2's Methods, as version 1 gives none. Contamination
above 0.5 is excluded, reading the Methods' "below 0.5" as a slip. The offsets include weakly tuned
cells; cells with half-width below 120 degrees are shown as a variant.

**What was found about the dandiset.** The archive's genotype of H3026 and date of birth of H3029
disagree with the paper's counts, which they match when H3026 is a juvenile Fmr1-/y rat and H3029 a
juvenile WT rat; the birth dates of H3023 and H3026 are after their recordings, which gives them
negative ages in the archive's asset metadata, and no NWB file sets the age. H4825's file holds
H4824's recording, re-timed to H4825's epochs and clipped at its end, so one adult Fmr1-/y rat's data
are missing. The units table has no cell type, quality or depth. The epoch and task tables mislabel or
merge sessions in several files and one file has a single epoch. Two files give positions in pixels
while stating centimetres, with head direction on another range; in the others y starts at exactly 0.
One unit has unsorted spike times, and juvenile files repeat tracking timestamps. The archive's
`about` field lists the Fmr1 protein under anatomy with the identifier of the disorder.